# Correlation Matrix calcuation method 3, the cvxpy method:

This is the 3rd potential method, this uses the same formulas and methods as the 2nd correlation method. Ie the one that is convex and performs projections to move matrixs to the proper subspace.

However, the module used here, cvxpy, is a symbolic module, meaning all the calucations are done symbolically and not numerically. This is a very very slow process and thusly we abandoned it as a method, however i leave here the code used for future reference.

You will note that there is no predictions, thats because the process of solving takes like 20 minutes, and at that point, having to tune hyperparameters is going to be so annoying, so why bother.

In [1]:
from tara_preprocessing import get_just_ecog_data,get_electrode_normalized_loc,car
from noah_production_funcs import single_patient_prediction_pure,create_lapaican_rbf
from tara_preprocessing import remove_duplicates, hold_out, preprocessing,apply_car_function,clip_time_series
from tara_preprocessing import make_patient_correlation_matrix
from noah_production_funcs import create_u
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import geoopt

In [3]:
data_root = Path("/Users/noahwanless/Desktop/Spring2026/M467/faces_basic/data")
registered_dir = Path("../SuperEeg-M467-project/registered_outputs")
ecogs = get_just_ecog_data(registered_dir,data_root)
xyz = get_electrode_normalized_loc(registered_dir)
print('Downloaded data')
ecogs = clip_time_series(ecogs)
print("Time series clipped")
ecogs_no_dups,xyz_no_dups = remove_duplicates(ecogs,xyz)
print('Removed duplicate electrodes')
xyz_clea, cleane = preprocessing(ecogs_no_dups,xyz_no_dups,notch_size=.05)
print("Done Preprocessing")
cleaned_f,xyz_f,fake_pat_beginning,held_out_elecs = hold_out(xyz_clea,cleane,0,[40,41])
cleaned_f = apply_car_function(cleaned_f,0)
print("Done holding out electrodes")
patient_corr_mat = make_patient_correlation_matrix(xyz_f,cleaned_f)
print('Got Correlation Matrices, done!')

[PosixPath('../SuperEeg-M467-project/registered_outputs/aa_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/ap_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/ca_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/de_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/fp_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/ha_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/ja_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/jm_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/jt_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/mv_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/rn_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_output

In [4]:
import cvxpy as cp


def object_func_2(K,C,L_sqrt,lamb,patient_node_num,num_pat):
    sum = 0
    iter = 0
    for i in range(num_pat):
        c = cp.Parameter((C[i].shape[0],C[i].shape[0])) #each patient correlation matrix
        c.value = C[i].to_numpy()
        num_nodes = patient_node_num[i]
        k = K[iter:iter+num_nodes,iter:iter+num_nodes] #all columns of rows and colums iter+num of nodes + 1 (this gets just that patients correlation matrix in K)
        diff = (k-c)
        sum = sum + (cp.norm(diff,p='fro'))**2
        iter = iter + num_nodes
    sum = sum + lamb*cp.trace(L_sqrt@K) #quad_form
    return sum

from scipy.linalg import sqrtm


def optmize_k(num_elec,object_func,C,L,lamb,patient_node_num,num_pat):
    #L = cp.Parameter(L)
    L_sqrt = np.real(sqrtm(L))
    L_sqrt_cp = cp.Parameter((L_sqrt.shape[0], L_sqrt.shape[1]),PSD=True)
    L_sqrt_cp.value = L_sqrt

    lamb_cp = cp.Parameter()
    lamb_cp.value= lamb
    K = cp.Variable((num_elec,num_elec))
    objective = cp.Minimize(object_func(K,C,L_sqrt_cp,lamb_cp,patient_node_num,num_pat))
    constraints = [K>>0, cp.diag(K) == 1]
    problem = cp.Problem(objective, constraints)
    print("problem is DCP:", problem.is_dcp())
    problem.solve(verbose=True)
    return K

In [5]:
L = create_lapaican_rbf(xyz_f,20)
num_pat = len(cleaned_f)
patient_node_num = []
for pat in cleaned_f:
    patient_node_num.append(pat.shape[1])
K = optmize_k(xyz_f.shape[0],object_func_2,patient_corr_mat,L,0.01,patient_node_num,num_pat)

(CVXPY) May 05 09:25:58 PM: Your problem has 320356 variables, 320922 constraints, and 348225 parameters.
(CVXPY) May 05 09:25:58 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) May 05 09:25:58 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) May 05 09:25:58 PM: Your problem is compiled with the CPP canonicalization backend.
/var/folders/8t/y5v94x215r973w80szwrht8r0000gq/T/ipykernel_8732/2750137310.py:34: UserWarning: You are solving a parameterized problem that is not DPP. Because the problem is not DPP, subsequent solves will not be faster than the first one. For more information, see the documentation on Disciplined Parametrized Programming, at https://www.cvxpy.org/tutorial/dpp/index.html
  problem.solve(verbose=True)
(CVXPY) May 05 09:25:58 PM: Compiling problem (target solver=SCS).
(CVXPY) May 05 09:25:58 PM: Reduction chain: EvalParams -> Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> SCS
(CV

problem is DCP: True
                                     CVXPY                                     
                                     v1.8.2                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) May 05 09:25:58 PM: Applying reduction SCS
(CVXPY) May 05 09:25:58 PM: Finished problem compilation (took 3.985e-01 seconds).
(CVXPY) May 05 09:25:58 PM: Invoking solver SCS  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
------------------------------------------------------------------
	       SCS v3.2.11 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 320371, constraints m: 188910
cones: 	  z: primal zero / dual free vars: 566
	  q: soc vars: 27883, qsize: 15
	  s: psd vars: 160461, ssize: 1
settings: eps_abs: 1.0e-05, eps_rel: 1.0e-05, eps_infeas: 1.0e-07
	  alpha: 1.50, scale: 1.00e-01, adaptive_scale: 1
	  max_iters: 100000, normalize: 1, rho_x: 1.00e-06
	  acceleration_lookback: 10, acceleration_interval: 10
lin-sys:  sparse-direct-amd-qdldl
	  nnz(A): 348805, nnz(P): 15
------------------------------------------------------------------
 iter | 

SolverError: Solver 'SCS' failed. Try another solver, or solve with verbose=True for more information.